# IDX-Trade — E2E Monte Carlo

Block-bootstrap historical **E2E session returns** for the frozen V4-X1 paper stack. Read-only; do not point this notebook at protected forward outcomes.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'idx_trade').is_dir())
sys.path.insert(0, str(REPO_ROOT / 'src'))

from idx_trade.v4_x1_decision_v1_contract import EXPECTED_ALPHA_MODEL_ID, EXPECTED_ALPHA_MODEL_FINGERPRINT
print(EXPECTED_ALPHA_MODEL_ID)
print(EXPECTED_ALPHA_MODEL_FINGERPRINT)


In [ ]:
# Historical/development E2E return or NAV export only.
DATA_PATH = None  # e.g. Path(r'D:\\...\\e2e_session_returns.parquet')

INITIAL_NAV = 50_000_000
N_PATHS = 10_000
HORIZON = 252
BLOCK = 5
SEED = 42
SESSIONS_PER_YEAR = 252


In [ ]:
RETURN_COLS = ['session_return', 'portfolio_return', 'return', 'ret']
RETURN_PCT_COLS = ['session_return_pct', 'portfolio_return_pct', 'return_pct']
NAV_COLS = ['total_return_nav_idr', 'nav_idr', 'portfolio_nav', 'nav']

def load_e2e_returns(path):
    path = Path(path)
    df = pd.read_parquet(path) if path.suffix.lower() == '.parquet' else pd.read_csv(path)
    lower = {str(c).lower(): c for c in df.columns}
    for name in RETURN_COLS:
        if name in lower:
            s = pd.to_numeric(df[lower[name]], errors='coerce').dropna(); break
    else:
        for name in RETURN_PCT_COLS:
            if name in lower:
                s = pd.to_numeric(df[lower[name]], errors='coerce').dropna() / 100; break
        else:
            for name in NAV_COLS:
                if name in lower:
                    nav = pd.to_numeric(df[lower[name]], errors='coerce').dropna()
                    s = nav.pct_change().dropna(); break
            else:
                raise ValueError('Need a session-return or NAV column.')
    s = s[np.isfinite(s)].reset_index(drop=True)
    if len(s) < max(20, BLOCK): raise ValueError(f'Too few sessions: {len(s)}')
    if (s <= -1).any(): raise ValueError('Return <= -100% found.')
    return s

if DATA_PATH is None:
    returns = None
    print('Set DATA_PATH, then rerun this cell.')
else:
    returns = load_e2e_returns(DATA_PATH)
    display(returns.describe(percentiles=[.05,.25,.5,.75,.95]).to_frame('session_return'))


In [ ]:
def block_bootstrap(x, n_paths=N_PATHS, horizon=HORIZON, block=BLOCK, seed=SEED):
    x = np.asarray(x, float)
    rng = np.random.default_rng(seed)
    out = np.empty((n_paths, horizon))
    max_start = len(x) - block
    if max_start < 0: raise ValueError('BLOCK is larger than the sample.')
    for j in range(0, horizon, block):
        width = min(block, horizon - j)
        starts = rng.integers(0, max_start + 1, size=n_paths)
        out[:, j:j+width] = x[starts[:, None] + np.arange(width)]
    return out

def max_drawdown(nav):
    peak = np.maximum.accumulate(nav, axis=1)
    return (nav / peak - 1).min(axis=1)

assert returns is not None, 'Set DATA_PATH first.'
sim_r = block_bootstrap(returns)
sim_nav = INITIAL_NAV * np.cumprod(1 + sim_r, axis=1)
terminal = sim_nav[:, -1]
mdd = max_drawdown(sim_nav)
cagr = (terminal / INITIAL_NAV) ** (SESSIONS_PER_YEAR / HORIZON) - 1

summary = pd.Series({
    'historical_sessions': len(returns),
    'historical_mean_return': returns.mean(),
    'historical_vol': returns.std(ddof=1),
    'historical_win_rate': (returns > 0).mean(),
    'terminal_p05': np.quantile(terminal, .05),
    'terminal_median': np.quantile(terminal, .50),
    'terminal_p95': np.quantile(terminal, .95),
    'median_cagr': np.median(cagr),
    'p_finish_below_start': np.mean(terminal < INITIAL_NAV),
    'median_max_drawdown': np.median(mdd),
    'p_drawdown_20pct': np.mean(mdd <= -.20),
    'p_drawdown_30pct': np.mean(mdd <= -.30),
})
display(summary.to_frame('value'))


In [ ]:
q = np.quantile(sim_nav, [.05,.25,.50,.75,.95], axis=0)
x = np.arange(1, HORIZON + 1)

plt.figure(figsize=(10,5))
plt.fill_between(x, q[0], q[4], alpha=.15)
plt.fill_between(x, q[1], q[3], alpha=.20)
plt.plot(x, q[2], linewidth=2, label='median')
plt.axhline(INITIAL_NAV, linewidth=1)
plt.title('Monte Carlo NAV fan')
plt.xlabel('session'); plt.ylabel('NAV (IDR)'); plt.legend(); plt.show()

plt.figure(figsize=(9,4))
plt.hist(terminal, bins=60)
plt.axvline(np.median(terminal), linewidth=2, label='median')
plt.axvline(INITIAL_NAV, linewidth=1, label='start')
plt.title(f'Terminal NAV — {HORIZON} sessions')
plt.xlabel('NAV (IDR)'); plt.ylabel('paths'); plt.legend(); plt.show()

pd.DataFrame({'terminal_nav': terminal, 'cagr': cagr, 'max_drawdown': mdd}).describe(percentiles=[.01,.05,.25,.5,.75,.95,.99])
